In [0]:
CREATE OR REPLACE TABLE paysim_fraud.gold.fraud_by_transaction_type AS

SELECT
    transaction_type,
    COUNT(*) AS transaction_count,
    SUM(amount) AS total_amount,
    AVG(amount) AS avg_amount,
    SUM(CASE WHEN is_fraud THEN 1 ELSE 0 END) AS fraud_count,
    SUM(CASE WHEN is_fraud THEN amount ELSE 0 END) AS fraud_amount,
    SUM(CASE WHEN is_flagged_fraud THEN 1 ELSE 0 END) AS flagged_count,
    AVG(CASE WHEN is_fraud THEN 1.0 ELSE 0.0 END) AS fraud_rate
FROM paysim_fraud.silver.transactions
GROUP BY transaction_type;

SELECT *
FROM paysim_fraud.gold.fraud_by_transaction_type
ORDER BY fraud_rate DESC;

In [0]:
CREATE OR REPLACE TABLE paysim_fraud.gold.fraud_metrics_hourly AS

SELECT
    step,
    simulation_day,
    hour_of_day,
    COUNT(*) AS transaction_count,
    SUM(amount) AS total_amount,
    AVG(amount) AS avg_transaction_amount,
    SUM(CASE WHEN is_fraud THEN 1 ELSE 0 END) AS fraud_count,
    SUM(CASE WHEN is_fraud THEN amount ELSE 0 END) AS fraud_amount,
    AVG(CASE WHEN is_fraud THEN 1.0 ELSE 0.0 END) AS fraud_rate,
    SUM(CASE WHEN is_flagged_fraud THEN 1 ELSE 0 END) AS flagged_count
FROM paysim_fraud.silver.transactions
GROUP BY
    step,
    simulation_day,
    hour_of_day;

In [0]:
CREATE OR REPLACE TABLE paysim_fraud.gold.rule_performance AS

SELECT
    SUM(CASE WHEN is_fraud AND is_flagged_fraud THEN 1 ELSE 0 END) AS true_positive,
    SUM(CASE WHEN NOT is_fraud AND is_flagged_fraud THEN 1 ELSE 0 END) AS false_positive,
    SUM(CASE WHEN is_fraud AND NOT is_flagged_fraud THEN 1 ELSE 0 END) AS false_negative,
    SUM(CASE WHEN NOT is_fraud AND NOT is_flagged_fraud THEN 1 ELSE 0 END) AS true_negative
FROM paysim_fraud.silver.transactions;

In [0]:
SELECT *
FROM paysim_fraud.gold.rule_performance;

In [0]:
CREATE OR REPLACE TABLE paysim_fraud.gold.fraud_by_amount_band AS

SELECT
    CASE
        WHEN amount < 1000 THEN '0-1K'
        WHEN amount < 10000 THEN '1K-10K'
        WHEN amount < 100000 THEN '10K-100K'
        WHEN amount < 500000 THEN '100K-500K'
        WHEN amount < 1000000 THEN '500K-1M'
        ELSE '1M+'
    END AS amount_band,

    COUNT(*) AS transaction_count,
    SUM(CASE WHEN is_fraud THEN 1 ELSE 0 END) AS fraud_count,
    SUM(amount) AS total_amount,
    SUM(CASE WHEN is_fraud THEN amount ELSE 0 END) AS fraud_amount,
    AVG(CASE WHEN is_fraud THEN 1.0 ELSE 0.0 END) AS fraud_rate

FROM paysim_fraud.silver.transactions

GROUP BY
    CASE
        WHEN amount < 1000 THEN '0-1K'
        WHEN amount < 10000 THEN '1K-10K'
        WHEN amount < 100000 THEN '10K-100K'
        WHEN amount < 500000 THEN '100K-500K'
        WHEN amount < 1000000 THEN '500K-1M'
        ELSE '1M+'
    END;

In [0]:
CREATE OR REPLACE TABLE paysim_fraud.gold.overall_metrics AS

SELECT
    COUNT(*) AS total_transactions,
    SUM(CASE WHEN is_fraud THEN 1 ELSE 0 END) AS fraud_transactions,
    AVG(CASE WHEN is_fraud THEN 1.0 ELSE 0.0 END) AS fraud_rate,
    SUM(amount) AS total_transaction_amount,
    SUM(CASE WHEN is_fraud THEN amount ELSE 0 END) AS fraud_amount
FROM paysim_fraud.silver.transactions;

In [0]:
SHOW TABLES IN paysim_fraud.gold;